# Timeseries classification with a Transformer model

**Author:** [Theodoros Ntakouris](https://github.com/ntakouris)<br>
**Anotated by (for CMPE 401):** [Musab Hassan](https://github.com/Musab-Hassan)<br>
**Date created:** 2021/06/25<br>
**Last modified:** 2026/03/12<br>
**Description:** This notebook demonstrates how to do timeseries classification using a Transformer model.

## Introduction

This is the Transformer architecture from
[Attention Is All You Need](https://arxiv.org/abs/1706.03762),
applied to timeseries instead of natural language.

This example requires TensorFlow 2.4 or higher.

## Load the dataset

We are going to use the same dataset and preprocessing as the
[TimeSeries Classification from Scratch](https://keras.io/examples/timeseries/timeseries_classification_from_scratch)
example.

In [1]:
import numpy as np
import keras
import gc
import pandas as pd
from keras import layers


def readucr(filename):
    data = np.loadtxt(filename, delimiter="\t")
    y = data[:, 0]
    x = data[:, 1:]
    return x, y.astype(int) 


root_url = "https://raw.githubusercontent.com/hfawaz/cd-diagram/master/FordA/"

x_train, y_train = readucr(root_url + "FordA_TRAIN.tsv")
x_test, y_test = readucr(root_url + "FordA_TEST.tsv")

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

n_classes = len(np.unique(y_train))

idx = np.random.permutation(len(x_train))
x_train = x_train[idx]
y_train = y_train[idx]

y_train[y_train == -1] = 0
y_test[y_test == -1] = 0

2026-03-23 22:51:27.950610: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 22:51:27.985397: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-23 22:51:29.266684: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Build the model

Our model processes a tensor of shape `(batch size, sequence length, features)`,
where `sequence length` is the number of time steps and `features` is each input
timeseries.

You can replace your classification RNN layers with this one: the
inputs are fully compatible!

We include residual connections, layer normalization, and dropout.
The resulting layer can be stacked multiple times.

The projection layers are implemented through `keras.layers.Conv1D`.

In [2]:
# This implementation applies Layer Normalization before the residual connection
# to improve training stability by producing better-behaved gradients and often
# eliminating the need for learning rate warm-up.


def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

The main part of our model is now complete. We can stack multiple of those
`transformer_encoder` blocks and we can also proceed to add the final
Multi-Layer Perceptron classification head. Apart from a stack of `Dense`
layers, we need to reduce the output tensor of the `TransformerEncoder` part of
our model down to a vector of features for each data point in the current
batch. A common way to achieve this is to use a pooling layer. For
this example, a `GlobalAveragePooling1D` layer is sufficient.

In [3]:
def build_model(
    input_shape,
    head_size,
    num_heads,
    ff_dim,
    num_transformer_blocks,
    mlp_units,
    dropout=0,
    mlp_dropout=0,
):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.GlobalAveragePooling1D(data_format="channels_last")(x)
    for dim in mlp_units:
        x = layers.Dense(dim, activation="relu")(x)
        x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)

## Train and evaluate

In [4]:
input_shape = x_train.shape[1:]

model = build_model(
    input_shape,
    head_size=256,
    num_heads=4,
    ff_dim=4,
    num_transformer_blocks=4,
    mlp_units=[128],
    mlp_dropout=0.4,
    dropout=0.25,
)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)
model.summary()

callbacks = [keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]

history_baseline = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=64,
    callbacks=callbacks,
)

model.evaluate(x_test, y_test, verbose=1)

I0000 00:00:1774331490.458606 1246766 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8323 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 500, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 500, 1)    │      7,169 │ input_layer[0][0… │
│ (MultiHeadAttentio… │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 500, 1)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 500, 1)    │          2 │ dropout_1[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 500, 4)    │          8 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 500, 4)    │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 500, 1)    │          5 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ conv1d_1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 500, 1)    │      7,169 │ add_1[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 500, 1)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ dropout_4[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 500, 4)    │          8 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 500, 4)    │          0 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 500, 1)    │          5 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ conv1d_3[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 29,258 (114.29 KB)

 Trainable params: 29,258 (114.29 KB)

...

2026-03-23 22:52:41.097598: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 16 bytes spill stores, 16 bytes spill loads

2026-03-23 22:52:41.390920: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_838', 20 bytes spill stores, 20 bytes spill loads

2026-03-23 22:52:41.468691: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_41', 4 bytes spill stores, 4 bytes spill loads



41/42 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.6929 - sparse_categorical_accuracy: 0.5305

2026-03-23 22:52:43.284636: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 16 bytes spill stores, 16 bytes spill loads



42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - loss: 0.6930 - sparse_categorical_accuracy: 0.5159


[0.6930360198020935, 0.5159090757369995]

## Modifications (Task 2)


In [5]:
# Store baseline results
test_acc_baseline = model.evaluate(x_test, y_test, verbose=0)[1]
val_acc_baseline = max(history_baseline.history['val_sparse_categorical_accuracy'])
experiments_results = [{"name": "Baseline", "num_blocks": 4, "num_heads": 4, "dropout": 0.25, "test_acc": test_acc_baseline, "val_acc": val_acc_baseline, "ff_dim": 4, "params": "Standard"}]
print(f"Baseline - Test Acc: {test_acc_baseline:.4f}, Val Acc: {val_acc_baseline:.4f}")

# Free memory
del model
keras.backend.clear_session()
gc.collect()

Baseline - Test Acc: 0.5159, Val Acc: 0.4868


0

In [6]:
# Modification 1: Moderately Reduced Transformers
model_v1 = build_model(
    input_shape,
    head_size=256,  # Same as baseline
    num_heads=3,  # Reduced from 4
    ff_dim=4,  # Same as baseline
    num_transformer_blocks=3,  # Reduced from 4
    mlp_units=[128],  # Same as baseline
    mlp_dropout=0.3,  # Reduced from 0.4
    dropout=0.2,  # Reduced from 0.25
)

model_v1.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)

history_v1 = model_v1.fit(x_train, y_train, validation_split=0.2, epochs=150, batch_size=64, callbacks=callbacks, verbose=0)
test_acc_v1 = model_v1.evaluate(x_test, y_test, verbose=0)[1]
val_acc_v1 = max(history_v1.history['val_sparse_categorical_accuracy'])

print(f"Mod 1 (Moderately Reduced Transformers) - Test Acc: {test_acc_v1:.4f}, Val Acc: {val_acc_v1:.4f}")
experiments_results.append({"name": "Mod 1: Moderately Reduced Transformers", "num_blocks": 3, "num_heads": 3, "dropout": 0.2, "test_acc": test_acc_v1, "val_acc": val_acc_v1, "ff_dim": 4})

2026-03-23 22:52:48.638819: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_59', 16 bytes spill stores, 16 bytes spill loads

2026-03-23 22:52:48.905555: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_77', 4 bytes spill stores, 4 bytes spill loads

2026-03-23 22:52:48.942162: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_59', 8 bytes spill stores, 8 bytes spill loads

2026-03-23 22:52:49.200442: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_65', 108 bytes spill stores, 108 bytes spill loads

2026-03-23 22:52:49.789683: I external/local_xla/x

Mod 1 (Moderately Reduced Transformers) - Test Acc: 0.5159, Val Acc: 0.4868


In [7]:
# Free memory
del model_v1
keras.backend.clear_session()
gc.collect()

0

In [8]:
# Modification 2: Small Model
model_v2 = build_model(
    input_shape,
    head_size=64,  # Reduced from 256
    num_heads=2,  # Reduced from 4
    ff_dim=2,  # Reduced from 4
    num_transformer_blocks=1,  # Reduced from 4
    mlp_units=[32],  # Reduced from 128
    mlp_dropout=0.4,
    dropout=0.25,
)

model_v2.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)

history_v2 = model_v2.fit(x_train, y_train, validation_split=0.2, epochs=150, batch_size=64, callbacks=callbacks, verbose=0)
test_acc_v2 = model_v2.evaluate(x_test, y_test, verbose=0)[1]
val_acc_v2 = max(history_v2.history['val_sparse_categorical_accuracy'])

print(f"Mod 2 (Small Model) - Test Acc: {test_acc_v2:.4f}, Val Acc: {val_acc_v2:.4f}")
experiments_results.append({"name": "Mod 2: Small Model", "ff_dim": 2, "test_acc": test_acc_v2, "val_acc": val_acc_v2, "num_blocks": 1, "num_heads": 2, "dropout": 0.25})

2026-03-23 22:53:31.664349: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 24 bytes spill stores, 24 bytes spill loads



Mod 2 (Small Model) - Test Acc: 0.5159, Val Acc: 0.4868


In [9]:
# Free memory
del model_v2
keras.backend.clear_session()
gc.collect()

0

In [10]:
# Modification 3: High Regularization
model_v3 = build_model(
    input_shape,
    head_size=256,
    num_heads=4,
    ff_dim=4,
    num_transformer_blocks=4,
    mlp_units=[128],
    mlp_dropout=0.6,  # Increased from 0.4
    dropout=0.5,  # Doubled from 0.25
)

model_v3.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)

history_v3 = model_v3.fit(x_train, y_train, validation_split=0.2, epochs=150, batch_size=64, callbacks=callbacks, verbose=0)
test_acc_v3 = model_v3.evaluate(x_test, y_test, verbose=0)[1]
val_acc_v3 = max(history_v3.history['val_sparse_categorical_accuracy'])
overfitting_gap_v3 = max(history_v3.history['sparse_categorical_accuracy']) - test_acc_v3

print(f"Mod 3 (High Regularization) - Test Acc: {test_acc_v3:.4f}, Val Acc: {val_acc_v3:.4f}, Overfitting Gap: {overfitting_gap_v3:.4f}")
experiments_results.append({"name": "Mod 3: High Regularization", "num_blocks": 4, "num_heads": 4, "dropout": 0.5, "ff_dim": 4, "test_acc": test_acc_v3, "val_acc": val_acc_v3})

2026-03-23 22:53:51.778783: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_7', 136 bytes spill stores, 116 bytes spill loads



Mod 3 (High Regularization) - Test Acc: 0.5159, Val Acc: 0.4868, Overfitting Gap: 0.0032


In [11]:
# Free memory
del model_v3
keras.backend.clear_session()
gc.collect()

0

In [12]:
# Display results summary
results_df = pd.DataFrame(experiments_results)
print(results_df[["name", "test_acc", "val_acc", "num_blocks", "num_heads", "dropout"]].to_string(index=False))

# Calculate and display insights
print("\nKEY INSIGHTS:")
print(f"Best Test Accuracy: {results_df['test_acc'].max():.4f} ({results_df.loc[results_df['test_acc'].idxmax(), 'name']})")
print(f"Worst Test Accuracy: {results_df['test_acc'].min():.4f} ({results_df.loc[results_df['test_acc'].idxmin(), 'name']})")
print(f"Accuracy Range: {results_df['test_acc'].max() - results_df['test_acc'].min():.4f}")
if 'val_acc' in results_df.columns:
    results_df['gap'] = results_df['test_acc'] - results_df['val_acc']
    print(f"\nVal-Test Gap (Overfitting Indicator):")
    for idx, row in results_df.iterrows():
        print(f"  {row['name']}: {row['gap']:.4f}")


                                  name  test_acc  val_acc  num_blocks  num_heads  dropout
                              Baseline  0.515909 0.486824           4          4     0.25
Mod 1: Moderately Reduced Transformers  0.515909 0.486824           3          3     0.20
                    Mod 2: Small Model  0.515909 0.486824           1          2     0.25
            Mod 3: High Regularization  0.515909 0.486824           4          4     0.50

KEY INSIGHTS:
Best Test Accuracy: 0.5159 (Baseline)
Worst Test Accuracy: 0.5159 (Baseline)
Accuracy Range: 0.0000

Val-Test Gap (Overfitting Indicator):
  Baseline: 0.0291
  Mod 1: Moderately Reduced Transformers: 0.0291
  Mod 2: Small Model: 0.0291
  Mod 3: High Regularization: 0.0291
